# AdventureWorks predictions on Databricks

This notebook uses a Kumo Relational model on Databricks Model Serving to answer business questions over the AdventureWorks sample tables in Unity Catalog.
There is no feature engineering and no training step.
You describe the tables, write each question as a predictive query, and the model answers from the data.

You will forecast product demand, find customers likely to stop ordering, recommend products, and predict a product attribute.
The last section asks questions through a Unity Catalog metric view.

The tables stay in Unity Catalog.
For each prediction, the client reads only the rows it needs with SQL and sends those rows to the endpoint.

**What you need**

- The four AdventureWorks tables in one schema: `adventureworks_customers`, `adventureworks_products`, `adventureworks_sales_order_headers` and `adventureworks_sales_order_details`.
- A Model Serving endpoint that serves a Kumo Relational model, and permission to query it.
- Serverless compute with environment version 4 or later, or Databricks Runtime 17.3 LTS or later.
  On an older runtime, from 14.3 LTS, set the `warehouse_id` widget to a SQL warehouse.
- Optionally, a metric view over these tables, for the metric view section.

## Setup

Run the next cell to create the widgets, fill them in, then run the rest.
The cell after it installs the client named in the `install` widget, from PyPI by default.

In [ ]:
dbutils.widgets.text(
    'install',
    'structured-data-models-client[relational,databricks,databricks-serving]',
    'pip install',
)
# To install wheels from a Unity Catalog volume instead, set `install` to:
# --find-links /Volumes/<catalog>/<schema>/<volume>/ structured-data-models-client[relational,databricks,databricks-serving]
dbutils.widgets.text('catalog', 'main', 'Catalog')
dbutils.widgets.text('schema', 'adventureworks', 'Schema')
dbutils.widgets.text('endpoint', '', 'Serving endpoint')
dbutils.widgets.text('warehouse_id', '', 'SQL warehouse ID (optional)')
dbutils.widgets.text('metric_view', '', 'Metric view (optional)')
dbutils.widgets.text('output_table', '', 'Output table (optional)')
dbutils.widgets.dropdown('ci_output', 'false', ['false', 'true'], 'CI output')

install = dbutils.widgets.get('install')

# Keep the runtime's NumPy, which its other packages are built against.
import numpy

keep_numpy = f'numpy=={numpy.__version__}'

In [ ]:
%pip install --only-binary=:all: $install $keep_numpy

In [ ]:
dbutils.library.restartPython()

Read the widgets, and check that the tables and the endpoint exist before going further.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from sdm_connectors import quote_ident

catalog = dbutils.widgets.get('catalog')
schema_name = dbutils.widgets.get('schema')
endpoint = dbutils.widgets.get('endpoint')
warehouse_id = dbutils.widgets.get('warehouse_id')
metric_view = dbutils.widgets.get('metric_view')
output_table = dbutils.widgets.get('output_table')

TABLES = {
    'customers': 'adventureworks_customers',
    'products': 'adventureworks_products',
    'sales_order_headers': 'adventureworks_sales_order_headers',
    'sales_order_details': 'adventureworks_sales_order_details',
}


def quoted(*parts):
    # Quote each part of a name, so that any catalog, schema or table name works in SQL.
    return '.'.join(quote_ident(part, char='`') for part in parts)


if not endpoint:
    raise ValueError(
        'Set the endpoint widget to a Kumo Relational serving endpoint.'
    )

found = {
    row.tableName
    for row in spark.sql(
        f'SHOW TABLES IN {quoted(catalog, schema_name)}'
    ).collect()
}
missing = sorted(set(TABLES.values()) - found)
if missing:
    raise ValueError(
        f'{catalog}.{schema_name} does not have {", ".join(missing)}. '
        'Load the AdventureWorks tables there, or set the catalog and schema '
        'widgets to the schema that has them.'
    )

workspace = WorkspaceClient()
try:
    state = workspace.serving_endpoints.get(endpoint).state
except NotFound:
    raise ValueError(
        f"There is no serving endpoint named '{endpoint}'."
    ) from None
if state and state.ready and state.ready.value == 'NOT_READY':
    raise ValueError(f"Serving endpoint '{endpoint}' is not ready.")
print(
    f'Found the four tables in {catalog}.{schema_name} and endpoint {endpoint}.'
)

## Connect to the tables

A `DatabricksSource` names the catalog and schema of the tables, and the connection that reads them.
By default it reads through this notebook's Spark session, as you.
If you set `warehouse_id`, it reads through that SQL warehouse instead.

In [ ]:
from sdm_connectors import DatabricksSource
from sdm_connectors.databricks import warehouse

if warehouse_id:
    connection = warehouse(warehouse_id, workspace=workspace)
else:
    connection = spark

source = DatabricksSource(connection, catalog=catalog, schema=schema_name)
source

## Prepare the schema

`Schema.from_source` describes each table from the catalog and a small preview of its rows.
Each table gets the short name used in the queries below.

These tables declare no primary or foreign keys in the catalog, so the cell states them.
It also names the date column of each table that records events.
`products` and `customers` get no time column, because their dates do not say when an event happened.
`SellStartDate` says when a product went on sale, and `ModifiedDate` says when a row was last edited.

In [ ]:
from sdm_client import Relationship, Schema

schema = Schema.from_source(
    source,
    TABLES,
    primary_keys={
        'customers': 'CustomerID',
        'products': 'ProductID',
        'sales_order_details': 'SalesOrderDetailID',
    },
    time_columns={
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
        'products': None,
        'customers': None,
    },
    relationships=[
        Relationship(
            'sales_order_headers', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'ProductID', 'products', 'ProductID'
        ),
    ],
)

`rowguid` is removed from every table.
It is a random ID per row, and an identifier that is different on every row carries no information the model can learn from.

In [ ]:
for table in schema.tables.values():
    if 'rowguid' in table:
        del table['rowguid']

for table in schema.tables.values():
    print(table)
print(schema.relationships)

## Choose what to score

Each prediction scores a list of IDs.
This cell picks the five best-selling products, the 25 customers with the most orders, and the ten products bought by the most customers.
It also reads product names and customer account numbers to label the results.

Some questions are asked at an anchor date 180 days before the last order.
A prediction at a past anchor uses only the data before that date, so you can compare it with what actually happened.
For those questions, the products and customers are picked from the orders up to the anchor, so that the choice does not depend on what happened after it.

In [ ]:
import pandas as pd


def table_name(name):
    return quoted(catalog, schema_name, TABLES[name])


details = table_name('sales_order_details')
headers = table_name('sales_order_headers')

latest = pd.Timestamp(
    spark.sql(f'SELECT max(OrderDate) FROM {details}').first()[0]
)
anchor = latest - pd.Timedelta(days=180)


def top_ids(table, key, order, limit, cutoff=None):
    # The `limit` IDs that rank highest by `order`, over every order or only those up to `cutoff`.
    where = f"WHERE OrderDate <= '{cutoff}'" if cutoff is not None else ''
    query = (
        f'SELECT {key} FROM {table} {where} GROUP BY {key} '
        f'ORDER BY {order} DESC, {key} LIMIT {limit}'
    )
    return [row[0] for row in spark.sql(query).collect()]


# For the questions asked at the last order.
top_products = top_ids(details, 'ProductID', 'sum(OrderQty)', 5)
frequent_customers = top_ids(headers, 'CustomerID', 'count(*)', 25)
# For the questions asked at the past anchor.
past_top_products = top_ids(details, 'ProductID', 'sum(OrderQty)', 5, anchor)
past_popular_products = top_ids(
    details, 'ProductID', 'count(DISTINCT CustomerID)', 10, anchor
)
past_frequent_customers = top_ids(headers, 'CustomerID', 'count(*)', 25, anchor)

labelled = sorted({*top_products, *past_top_products, *past_popular_products})
products = spark.sql(
    f'SELECT ProductID, Name, Color FROM {table_name("products")} '
    f'WHERE ProductID IN ({", ".join(map(str, labelled))})'
).toPandas()
customers = spark.sql(
    f'SELECT CustomerID AS id, AccountNumber FROM {table_name("customers")} '
    f'WHERE CustomerID IN ({", ".join(map(str, frequent_customers))})'
).toPandas()
names = products.set_index('ProductID')['Name']

print(f'Last order: {latest.date()}. Past anchor: {anchor.date()}.')
display(products[products['ProductID'].isin(top_products)])

## Connect to the endpoint

`SDMClient.for_databricks_serving` calls the endpoint as you, so there is no token or URL to set.
If the endpoint has scaled to zero, the first call takes a few minutes while it starts.

In [ ]:
from sdm_client import KumoRelational, PredictiveQuery, SDMClient

client = SDMClient.for_databricks_serving(endpoint)
model = KumoRelational()
predictions = {}

## Demand forecast

How many units of each top product will sell in the next 30 days?

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)   ← units sold in the 30 days after the anchor
FOR EACH products.ProductID                              ← one prediction per product
```

With no anchor, the forecast starts from the last order in the data.
`last_30_days` is the number of units each product sold in the 30 days up to the last order, for comparison.
The next section asks the same question at a past anchor, where the forecast can be checked against actual sales.

In [ ]:
def actual_sales(start, days=30, column='OrderQty'):
    # The sum of `column` per product over the same window as the query: after `start`, up to and including its last day.
    end = start + pd.Timedelta(days=days)
    return spark.sql(
        f'SELECT ProductID AS id, sum({column}) AS actual FROM {details} '
        f"WHERE OrderDate > '{start}' AND OrderDate <= '{end}' GROUP BY ProductID"
    ).toPandas()


DEMAND = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FOR EACH products.ProductID'
)

request = PredictiveQuery(DEMAND, schema).materialize(indices=top_products)
demand = client.predict(request, model=model).to_pandas()

recent = actual_sales(latest - pd.Timedelta(days=30))
demand = demand.merge(
    recent.rename(columns={'actual': 'last_30_days'}), on='id', how='left'
)
demand['last_30_days'] = demand['last_30_days'].fillna(0)
demand['product'] = demand['id'].map(names)
predictions['demand_30d'] = demand
display(demand[['id', 'product', 'prediction', 'last_30_days']])

## Demand forecast at a past date

The same question, asked at the past anchor.
The model sees only the data before the anchor, so the forecast can be compared with the units that sold in the 30 days after it.

In [ ]:
import matplotlib.pyplot as plt

request = PredictiveQuery(DEMAND, schema).materialize(
    indices=past_top_products, anchor_time=anchor
)
past_demand = client.predict(request, model=model).to_pandas()

past_demand = past_demand.merge(actual_sales(anchor), on='id', how='left')
past_demand['actual'] = past_demand['actual'].fillna(0)
past_demand['product'] = past_demand['id'].map(names)
predictions['demand_30d_past'] = past_demand
display(past_demand[['id', 'product', 'prediction', 'actual']])

past_demand.plot.bar(
    x='product',
    y=['prediction', 'actual'],
    rot=20,
    title=f'Units sold in the 30 days after {anchor.date()}',
)
plt.show()

## Multi-period forecast

Forecast the next three 30-day windows after the past anchor, and compare each with actual sales.

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)   ← units sold in a 30-day window
FORECAST 3 TIMEFRAMES                                    ← three windows in a row
FOR EACH products.ProductID
```

The client asks one question per window, so this makes three calls to the endpoint.
`forecast_step` numbers the windows from 1.

In [ ]:
request = PredictiveQuery(
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FORECAST 3 TIMEFRAMES FOR EACH products.ProductID',
    schema,
).materialize(indices=past_top_products, anchor_time=anchor)
forecast = client.predict(request, model=model).to_pandas()

actuals = pd.concat(
    actual_sales(anchor + pd.Timedelta(days=30 * (step - 1))).assign(
        forecast_step=step
    )
    for step in (1, 2, 3)
)
forecast = forecast.merge(actuals, on=['id', 'forecast_step'], how='left')
forecast['actual'] = forecast['actual'].fillna(0)
forecast['product'] = forecast['id'].map(names)
predictions['demand_3_windows'] = forecast
display(
    forecast.sort_values(['product', 'forecast_step'])[
        ['id', 'product', 'forecast_step', 'prediction', 'actual']
    ]
)

## Customer churn

Which frequent customers will place no order in the next 90 days?

```
PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0   ← no orders in the next 90 days
FOR EACH customers.CustomerID                           ← one prediction per customer
```

This is a yes-or-no question, so the result has a probability for each answer.
`probability_True` is the probability that the customer places no order.
The table is sorted by it, highest first.

In [ ]:
request = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0 '
    'FOR EACH customers.CustomerID',
    schema,
).materialize(indices=frequent_customers)
churn = client.predict(request, model=model).to_pandas()

churn = churn.merge(customers, on='id', how='left')
churn = churn.sort_values('probability_True', ascending=False)
predictions['churn_90d'] = churn
display(churn[['id', 'AccountNumber', 'probability_True', 'prediction']])

## Product recommendations

Which products will a customer buy in the next 30 days?

```
PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days)   ← products bought in the next 30 days
RANK TOP 5                                                          ← the five most likely
FOR EACH customers.CustomerID                                       ← one list per customer
```

A ranking chooses from a list of candidates.
Here they are the ten products bought by the most customers up to the past anchor.
The client asks one question per candidate, so this makes up to ten calls.
The question is asked at the past anchor, for three of the most frequent customers up to that date.

A candidate that no customer bought within the window, in the sampled context, cannot be scored.
It is left out of the rankings and listed in `unscored_candidates`.

In [ ]:
request = PredictiveQuery(
    'PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days) '
    'RANK TOP 5 FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=past_frequent_customers[:3],
    candidates=past_popular_products,
    anchor_time=anchor,
)
result = client.predict(request, model=model)
recommendations = result.to_pandas()
predictions['recommend_top5'] = recommendations

print('Unscored candidates:', list(result.unscored_candidates) or 'none')
display(
    pd.DataFrame(
        {
            'customer': row.id,
            'rank': rank,
            'product': names[item['candidate']],
            'score': item['score'],
        }
        for row in recommendations.itertuples()
        for rank, item in enumerate(row.rankings, start=1)
    )
)

## Attribute prediction

What color is each product?
There is no time window, so this predicts a property of the product rather than a future event.

```
PREDICT products.Color        ← the product's color
FOR EACH products.ProductID   ← one prediction per product
```

`Color` takes a few values, so this is a classification.
The result has the predicted color and a probability for each color.
The recorded color is shown for comparison.
On your own data, the same query fills in values that are missing.

In [ ]:
request = PredictiveQuery(
    'PREDICT products.Color FOR EACH products.ProductID', schema
).materialize(indices=top_products)
colors = client.predict(request, model=model).to_pandas()

colors['product'] = colors['id'].map(names)
colors['recorded'] = colors['id'].map(products.set_index('ProductID')['Color'])
predictions['product_color'] = colors
probabilities = [
    column for column in colors if column.startswith('probability_')
]
display(colors[['id', 'product', 'recorded', 'prediction', *probabilities]])

## Metric views

A Unity Catalog metric view defines a source table, the tables it joins, and dimensions and measures over them.
`source.semantic_model(name)` returns a metric view as a source of its own, and `Schema.from_source` reads it into a schema with its tables, keys and links.
You can then ask questions through the view without describing the tables yourself.

Set the `metric_view` widget to run this section.
Give a full name such as `my-catalog.adventureworks.adventureworks_sales_metrics`, or just the view name to look it up in the catalog and schema widgets.

This notebook was tested with a view created by the statement below.
Replace `main.adventureworks` with your catalog and schema, and put a name with a hyphen in backticks, such as `` `my-catalog` ``.

```sql
CREATE VIEW main.adventureworks.adventureworks_sales_metrics
WITH METRICS LANGUAGE YAML AS $$
version: 0.1
source: main.adventureworks.adventureworks_sales_order_details
joins:
  - name: customers
    source: main.adventureworks.adventureworks_customers
    on: source.CustomerID = customers.CustomerID
  - name: products
    source: main.adventureworks.adventureworks_products
    on: source.ProductID = products.ProductID
dimensions:
  - name: SalesOrderDetailID
    expr: source.SalesOrderDetailID
  - name: SalesOrderID
    expr: source.SalesOrderID
  - name: OrderDate
    expr: source.OrderDate
  - name: OrderQty
    expr: source.OrderQty
  - name: UnitPrice
    expr: source.UnitPrice
  - name: LineTotal
    expr: source.LineTotal
  - name: Discounted
    expr: source.UnitPriceDiscount > 0
  - name: TerritoryID
    expr: customers.TerritoryID
  - name: StoreID
    expr: customers.StoreID
  - name: Color
    expr: products.Color
  - name: ListPrice
    expr: products.ListPrice
  - name: ProductLine
    expr: products.ProductLine
  - name: ProductSubcategoryID
    expr: products.ProductSubcategoryID
measures:
  - name: total_quantity
    expr: SUM(source.OrderQty)
  - name: revenue
    expr: SUM(source.LineTotal)
  - name: order_count
    expr: COUNT(DISTINCT source.SalesOrderID)
$$
```

The source table keeps its own name, `adventureworks_sales_order_details`, and each join becomes a table named after the join.
Dimensions become columns.
Measures are aggregations rather than columns, so the conversion notes give the predictive query aggregation that matches each one.

In [ ]:
view_schema = None
if metric_view:
    view_schema = Schema.from_source(source.semantic_model(metric_view))
    for table in view_schema.tables.values():
        print(table)
    print(view_schema.relationships)
    print(*view_schema.conversion_notes, sep='\n')
else:
    print('Set the metric_view widget to run this section.')

Ask the demand forecast at the past anchor through the view, and compare it with the forecast from the tables and with actual sales.
The view exposes less data than the tables, with no `sales_order_headers` and fewer columns, so its forecast can differ from the one from the tables.

In [ ]:
if view_schema is not None:
    request = PredictiveQuery(
        'PREDICT SUM(adventureworks_sales_order_details.OrderQty, 0, 30, days) '
        'FOR EACH products.ProductID',
        view_schema,
    ).materialize(indices=past_top_products, anchor_time=anchor)
    view_demand = client.predict(request, model=model).to_pandas()

    view_demand = view_demand.merge(
        past_demand[['id', 'product', 'prediction', 'actual']],
        on='id',
        suffixes=('_view', '_tables'),
    )
    predictions['view_demand_30d_past'] = view_demand
    display(
        view_demand[
            ['id', 'product', 'prediction_view', 'prediction_tables', 'actual']
        ]
    )

The view's `revenue` measure is `SUM(source.LineTotal)`.
Over the 30 days after the anchor, that is the query below.
The result is shown next to the actual revenue in that window.

In [ ]:
if view_schema is not None:
    request = PredictiveQuery(
        'PREDICT SUM(adventureworks_sales_order_details.LineTotal, 0, 30, days) '
        'FOR EACH products.ProductID',
        view_schema,
    ).materialize(indices=past_top_products, anchor_time=anchor)
    view_revenue = client.predict(request, model=model).to_pandas()

    view_revenue = view_revenue.merge(
        actual_sales(anchor, column='LineTotal'), on='id', how='left'
    )
    view_revenue['actual'] = view_revenue['actual'].fillna(0)
    view_revenue['product'] = view_revenue['id'].map(names)
    predictions['view_revenue_30d_past'] = view_revenue
    display(view_revenue[['id', 'product', 'prediction', 'actual']])

## Save predictions

To keep the predictions in a table you can query later, set the `output_table` widget.
Every prediction above is then appended to that Delta table.
Give a table name, `schema.table` or `catalog.schema.table`.
A name without a catalog or schema uses the ones in the widgets.
Each row holds the question name and one predicted entity as JSON, because the questions return different columns.

In [ ]:
if output_table:
    from pyspark.sql.functions import current_timestamp

    parts = output_table.strip().split('.')
    if len(parts) > 3 or not all(parts):
        raise ValueError(
            'Set output_table to table, schema.table or catalog.schema.table, '
            f"not '{output_table}'."
        )
    # Take the parts the name leaves out from the catalog and schema widgets.
    target = quoted(*[catalog, schema_name][: 3 - len(parts)], *parts)
    rows = [
        (question, line)
        for question, frame in predictions.items()
        for line in frame.to_json(orient='records', lines=True).splitlines()
    ]
    (
        spark.createDataFrame(rows, 'question string, prediction string')
        .withColumn('scored_at', current_timestamp())
        .write.mode('append')
        .saveAsTable(target)
    )
    print(f'Appended {len(rows)} rows to {target}.')
else:
    print('Set the output_table widget to save the predictions.')

## Clean up

Close the client, and the warehouse connection if the notebook opened one.

In [ ]:
client.close()
if warehouse_id:
    connection.close()

## Use your own data

To ask these questions of your own tables, change three things:

1. **The tables.**
   In `Schema.from_source`, map a short name to each table, and state the primary keys and relationships the catalog does not declare.
2. **The time columns.**
   Name the date column of each table that records events, and pass `None` for tables that describe things, like `products` and `customers` here.
   Remove columns that only identify a row, such as random IDs.
3. **The questions.**
   Write a predictive query for each question, and pass the IDs to score as `indices`.

## Troubleshooting

**The first prediction is slow or times out.**
An endpoint with scale to zero needs a few minutes to start after it has been idle.
The client waits up to five minutes per call.
Run the cell again once the endpoint is up.

**`... is not a supported wheel on this platform` during install.**
The relational engine is compiled, with separate wheels for x86_64 and aarch64 Linux, and serverless compute runs on either.
Install from PyPI, or put the wheels for both architectures in the folder you pass to `--find-links`.

**A request is refused as too large.**
Model Serving accepts requests of up to 16 MB, and the client checks the size before sending.
Score fewer IDs per call, or pass a smaller `num_context_samples` to `materialize`.

**A `StructuredDataError`.**
The client checks each query against the schema and the endpoint's limits, and refuses one it cannot send.
The message says what is wrong and what to change, for example a column the schema does not have, or a ranking where no candidate could be scored.

## Test output

This cell is for automated test runs.
With the `ci_output` widget set to `true`, it prints the environment and ends the run with every prediction as JSON.

In [ ]:
if dbutils.widgets.get('ci_output') == 'true':
    import json
    import os
    import platform
    import sys
    from importlib.metadata import version

    environment = {
        'runtime': os.environ.get('DATABRICKS_RUNTIME_VERSION'),
        'python': sys.version.split()[0],
        'machine': platform.machine(),
        'client': version('structured-data-models-client'),
        'engine': version('structured-data-models-relational-engine'),
    }
    print(environment)
    notes = view_schema.conversion_notes if view_schema is not None else []
    dbutils.notebook.exit(
        json.dumps(
            {
                'environment': environment,
                'results': {
                    name: frame.to_dict('records')
                    for name, frame in predictions.items()
                },
                'view_notes': [str(note) for note in notes],
            },
            default=str,
        )
    )